In [ ]:
import numpy as np
import pickle
import os

# 1. Yerel Dosyadan Veri Yükleme Fonksiyonu
def load_cifar10_batch(file):
    with open(file, 'rb') as fo:
        dict = pickle.load(fo, encoding='bytes')
    X = dict[b'data']
    y = dict[b'labels']
    return X, np.array(y)

def load_local_cifar10(root_path):
    # Eğitim verisi 5 parçadan oluşur, hepsini birleştirelim
    xs = []
    ys = []
    for b in range(1, 6):
        f = os.path.join(root_path, f'data_batch_{b}')
        X, Y = load_cifar10_batch(f)
        xs.append(X)
        ys.append(Y)
    
    X_train = np.concatenate(xs)
    y_train = np.concatenate(ys)
    
    # Test verisini yükle
    X_test, y_test = load_cifar10_batch(os.path.join(root_path, 'test_batch'))
    
    return (X_train, y_train), (X_test, y_test)

# --- ANA PROGRAM ---

# Veri setinin bulunduğu klasör yolu (Örn: "cifar-10-batches-py")
data_path = 'cifar-10-batches-py'

print("Yerel klasörden veriler yükleniyor...")
try:
    (X_train_raw, y_train_raw), (X_test_raw, y_test_raw) = load_local_cifar10(data_path)
except FileNotFoundError:
    print(f"Hata: '{data_path}' klasörü bulunamadı! Lütfen verileri klasöre ekleyin.")
    exit()

# Hız için örneklem alalım
train_size = 2000
test_size = 200

X_train = X_train_raw[:train_size].astype("float32") / 255.0
y_train = y_train_raw[:train_size]
X_test = X_test_raw[:test_size].astype("float32") / 255.0
y_test = y_test_raw[:test_size]

# 2. Mesafe Fonksiyonları
def compute_l1_distances(X_train, X_test):
    num_test = X_test.shape[0]
    num_train = X_train.shape[0]
    dists = np.zeros((num_test, num_train))
    for i in range(num_test):
        dists[i, :] = np.sum(np.abs(X_train - X_test[i, :]), axis=1)
    return dists

def compute_l2_distances(X_train, X_test):
    num_test = X_test.shape[0]
    num_train = X_train.shape[0]
    dists = np.zeros((num_test, num_train))
    for i in range(num_test):
        dists[i, :] = np.sqrt(np.sum(np.square(X_train - X_test[i, :]), axis=1))
    return dists

# 3. Tahmin Fonksiyonu
def predict(dists, y_train, k):
    num_test = dists.shape[0]
    y_pred = np.zeros(num_test)
    for i in range(num_test):
        closest_indices = np.argsort(dists[i])[:k]
        closest_y = y_train[closest_indices]
        y_pred[i] = np.argmax(np.bincount(closest_y.astype('int64')))
    return y_pred

# 4. Deney ve Karşılaştırma
k_values = [2, 5, 7]
print("Mesafeler hesaplanıyor (Bu işlem biraz sürebilir)...")
metrics = {
    "L1 (Manhattan)": compute_l1_distances(X_train, X_test),
    "L2 (Euclidean)": compute_l2_distances(X_train, X_test)
}

print("-" * 40)
for metric_name, dist_matrix in metrics.items():
    print(f"Metot: {metric_name}")
    for k in k_values:
        y_pred = predict(dist_matrix, y_train, k=k)
        accuracy = np.mean(y_pred == y_test)
        print(f"  k={k} için Doğruluk: %{accuracy * 100:.2f}")
    print("-" * 40)

In [3]:
import numpy as np
from tensorflow.keras.datasets import cifar10

# 1. Verileri Yükleme ve Hazırlama
print("Veriler yükleniyor...")
(X_train_raw, y_train_raw), (X_test_raw, y_test_raw) = cifar10.load_data()

# Hesaplama hızı için veri setinden küçük bir örneklem alalım
# (Örn: 2000 eğitim, 200 test resmi)
train_size = 2000
test_size = 200

X_train = X_train_raw[:train_size].reshape(train_size, -1).astype("float32") / 255.0
y_train = y_train_raw[:train_size].flatten()
X_test = X_test_raw[:test_size].reshape(test_size, -1).astype("float32") / 255.0
y_test = y_test_raw[:test_size].flatten()

# 2. Mesafe Fonksiyonları (Açık Yazım)

def compute_l1_distances(X_train, X_test):
    """Manhattan Mesafesi: sum(|x1 - x2|)"""
    num_test = X_test.shape[0]
    num_train = X_train.shape[0]
    dists = np.zeros((num_test, num_train))
    
    for i in range(num_test):
        # Test resmi ile tüm eğitim resimleri arasındaki mutlak farkların toplamı
        dists[i, :] = np.sum(np.abs(X_train - X_test[i, :]), axis=1)
    return dists

def compute_l2_distances(X_train, X_test):
    """Öklid Mesafesi: sqrt(sum((x1 - x2)^2))"""
    num_test = X_test.shape[0]
    num_train = X_train.shape[0]
    dists = np.zeros((num_test, num_train))
    
    for i in range(num_test):
        # Test resmi ile tüm eğitim resimleri arasındaki farkın karesinin karekökü
        dists[i, :] = np.sqrt(np.sum(np.square(X_train - X_test[i, :]), axis=1))
    return dists

# 3. Tahmin Fonksiyonu
def predict(dists, y_train, k):
    num_test = dists.shape[0]
    y_pred = np.zeros(num_test)
    for i in range(num_test):
        # En yakın k mesafeye sahip olanların indekslerini bul
        closest_indices = np.argsort(dists[i])[:k]
        closest_y = y_train[closest_indices]
        # Oylama yap (en çok tekrar eden sınıf)
        y_pred[i] = np.argmax(np.bincount(closest_y.astype('int64')))
    return y_pred

# 4. Deney ve Karşılaştırma
k_values = [2, 5, 7]
metrics = {
    "L1 (Manhattan)": compute_l1_distances(X_train, X_test),
    "L2 (Euclidean)": compute_l2_distances(X_train, X_test)
}

print("-" * 40)
for metric_name, dist_matrix in metrics.items():
    print(f"Metot: {metric_name}")
    for k in k_values:
        y_pred = predict(dist_matrix, y_train, k=k)
        accuracy = np.mean(y_pred == y_test)
        print(f"  k={k} için Doğruluk: %{accuracy * 100:.2f}")
    print("-" * 40)

Veriler yükleniyor...
----------------------------------------
Metot: L1 (Manhattan)
  k=2 için Doğruluk: %20.50
  k=5 için Doğruluk: %27.50
  k=7 için Doğruluk: %24.50
----------------------------------------
Metot: L2 (Euclidean)
  k=2 için Doğruluk: %20.00
  k=5 için Doğruluk: %22.50
  k=7 için Doğruluk: %24.50
----------------------------------------
